In [ ]:
# =========================================
# Bayesian Regression
# =========================================

import os
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import jax

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================
# STEP 1: LOAD DATA
# =========================================
path = "../data/indian_roads_dataset.csv"
df = pd.read_csv(path)

# =========================================
# STEP 2: PREPROCESSING
# =========================================
df['festival'] = df['festival'].notna().astype(int)

day_map = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2,
    'Thursday': 3, 'Friday': 4,
    'Saturday': 5, 'Sunday': 6
}

df['weekday_num'] = df['day_of_week'].map(day_map)

df['weekday_cos'] = np.cos(2 * np.pi * df['weekday_num'] / 7)
df['weekday_sin'] = np.sin(2 * np.pi * df['weekday_num'] / 7)

df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)

one_hot_cols = [
    'city', 'state', 'road_type',
    'traffic_density', 'weather',
    'visibility'
]

scale_cols = [
    'latitude', 'longitude',
    'lanes', 'temperature',
    'vehicles_involved'
]

df_encoded = pd.get_dummies(
    df,
    columns=one_hot_cols,
    drop_first=True
)

scaler = StandardScaler()
df_encoded[scale_cols] = scaler.fit_transform(
    df_encoded[scale_cols]
)

df_encoded.drop(columns=['weekday_num'], inplace=True)

# =========================================
# STEP 3: FEATURES + TARGET
# =========================================
y = df_encoded['risk_score']

X = df_encoded.drop(columns=[
    'date', 'time', 'hour',
    'day_of_week', 'accident_id',
    'risk_score', 'cause',
    'accident_severity', 'casualties'
])

# =========================================
# STEP 4: TRAIN TEST SPLIT
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train_np = X_train.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)

y_train_np = y_train.values.astype(np.float32)
y_test_np = y_test.values.astype(np.float32)

print("GPU:", jax.devices())

# =========================================
# MODEL 1: BAYESIAN LINEAR REGRESSION
# =========================================
with pm.Model() as bayesian_regression:

    intercept = pm.Normal("intercept", mu=0, sigma=10)

    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=10,
        shape=X_train_np.shape[1]
    )

    sigma = pm.HalfNormal("sigma", sigma=1)

    mu = intercept + pm.math.dot(X_train_np, beta)

    y_obs = pm.Normal(
        "y_obs",
        mu=mu,
        sigma=sigma,
        observed=y_train_np
    )

    trace_linear = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        target_accept=0.9,
        nuts_sampler="numpyro",
        random_seed=42
    )

intercept_mean = trace_linear.posterior["intercept"].mean().values
beta_mean = trace_linear.posterior["beta"].mean(
    dim=("chain", "draw")
).values

y_pred_linear = intercept_mean + np.dot(X_test_np, beta_mean)

print("\n===== MODEL 1: BAYESIAN LINEAR REGRESSION =====")
print("MAE :", mean_absolute_error(y_test_np, y_pred_linear))
print("RMSE:", np.sqrt(mean_squared_error(y_test_np, y_pred_linear)))
print("R2  :", r2_score(y_test_np, y_pred_linear))
print(az.summary(trace_linear))

# =========================================
# MODEL 2: SIMPLE BETA REGRESSION
# =========================================
eps = 1e-5
y_train_beta = np.clip(y_train_np, eps, 1 - eps)
y_test_beta = np.clip(y_test_np, eps, 1 - eps)

n_features = X_train_np.shape[1]

with pm.Model() as beta_regression_model:

    intercept = pm.Normal("intercept", mu=0, sigma=2)

    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=1,
        shape=n_features
    )

    phi = pm.HalfNormal("phi", sigma=10)

    eta = intercept + pm.math.dot(X_train_np, beta)

    mu = pm.Deterministic(
        "mu",
        pm.math.sigmoid(eta)
    )

    alpha = mu * phi
    beta_param = (1 - mu) * phi

    y_obs = pm.Beta(
        "y_obs",
        alpha=alpha,
        beta=beta_param,
        observed=y_train_beta
    )

    trace_beta = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.95,
        nuts_sampler="numpyro",
        random_seed=42
    )

intercept_mean = trace_beta.posterior["intercept"].mean().values
beta_mean = trace_beta.posterior["beta"].mean(
    dim=("chain", "draw")
).values

eta_test = intercept_mean + np.dot(X_test_np, beta_mean)
y_pred_beta = 1 / (1 + np.exp(-eta_test))

print("\n===== MODEL 2: SIMPLE BETA REGRESSION =====")
print("MAE :", mean_absolute_error(y_test_beta, y_pred_beta))
print("RMSE:", np.sqrt(mean_squared_error(y_test_beta, y_pred_beta)))
print("R2  :", r2_score(y_test_beta, y_pred_beta))
print(az.summary(trace_beta))

# =========================================
# MODEL 3: IMPROVED BETA REGRESSION
# =========================================
with pm.Model() as improved_beta_regression_model:

    intercept = pm.Normal("intercept", mu=0, sigma=1)

    tau = pm.HalfNormal("tau", sigma=1)

    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=tau,
        shape=n_features
    )

    eta = intercept + pm.math.dot(X_train_np, beta)

    mu = pm.Deterministic(
        "mu",
        pm.math.sigmoid(eta)
    )

    gamma_intercept = pm.Normal(
        "gamma_intercept",
        mu=0,
        sigma=1
    )

    gamma = pm.Normal(
        "gamma",
        mu=0,
        sigma=0.5,
        shape=n_features
    )

    log_phi = gamma_intercept + pm.math.dot(
        X_train_np,
        gamma
    )

    phi = pm.Deterministic(
        "phi",
        pm.math.exp(log_phi)
    )

    alpha = mu * phi
    beta_param = (1 - mu) * phi

    y_obs = pm.Beta(
        "y_obs",
        alpha=alpha,
        beta=beta_param,
        observed=y_train_beta
    )

    trace_improved = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,
        target_accept=0.98,
        nuts_sampler="numpyro",
        random_seed=42
    )

posterior = trace_improved.posterior.stack(
    sample=("chain", "draw")
)

intercept_samples = posterior["intercept"].values
beta_samples = posterior["beta"].values.T

eta_samples = (
    intercept_samples[:, None]
    + np.dot(beta_samples, X_test_np.T)
)

y_pred_improved = (
    1 / (1 + np.exp(-eta_samples))
).mean(axis=0)

print("\n===== MODEL 3: IMPROVED BETA REGRESSION =====")
print("MAE :", mean_absolute_error(y_test_beta, y_pred_improved))
print("RMSE:", np.sqrt(mean_squared_error(y_test_beta, y_pred_improved)))
print("R2  :", r2_score(y_test_beta, y_pred_improved))
print(az.summary(trace_improved))

print("\nDivergences:",
      trace_improved.sample_stats["diverging"].sum().values)

In [ ]:
# =========================================
# Bayesian Classification
# =========================================

# =========================================
# STEP 1: IMPORTS
# =========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

# =========================================
# STEP 2: LOAD DATA
# =========================================
path = "../data/indian_roads_dataset.csv"
df = pd.read_csv(path)


# =========================================
# STEP 3: PREPROCESSING
# =========================================

# festival binary
df['festival'] = df['festival'].notna().astype(int)

# weekday mapping
day_map = {
    'Monday': 0,
    'Tuesday': 1,
    'Wednesday': 2,
    'Thursday': 3,
    'Friday': 4,
    'Saturday': 5,
    'Sunday': 6
}

df['weekday_num'] = df['day_of_week'].map(day_map)

# cyclical encoding
df['weekday_cos'] = np.cos(2 * np.pi * df['weekday_num'] / 7)
df['weekday_sin'] = np.sin(2 * np.pi * df['weekday_num'] / 7)

df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)

# one hot columns
one_hot = [
    'city', 'state', 'road_type',
    'traffic_density', 'weather',
    'visibility', 'cause'
]

# scale columns
scale = [
    'latitude', 'longitude',
    'lanes', 'temperature',
    'vehicles_involved', 'casualties'
]

# one hot encoding
df_encoded = pd.get_dummies(df, columns=one_hot, drop_first=True)

# scaling
scaler = StandardScaler()
df_encoded[scale] = scaler.fit_transform(df_encoded[scale])

# drop helper
df_encoded.drop(columns=['weekday_num'], inplace=True)

# target + features
y = df_encoded['severity_class']

X = df_encoded.drop(columns=[
    'date', 'time', 'hour',
    'day_of_week', 'accident_id',
    'severity_class',
    'accident_severity'
])

print("\nProcessed Shape:", X.shape)

# =========================================
# COMMON TRAIN TEST SPLIT
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain Distribution:")
print(Counter(y_train))

# =========================================
# MODEL 1: NAIVE BAYES + SMOTE
# =========================================
print("\n" + "="*60)
print("MODEL 1: NAIVE BAYES + SMOTE")
print("="*60)

smote = SMOTE(
    sampling_strategy={2: 4500},
    random_state=42
)

X_nb, y_nb = smote.fit_resample(X_train, y_train)

nb_model = GaussianNB()
nb_model.fit(X_nb, y_nb)

y_pred_nb = nb_model.predict(X_test)
y_prob_nb = nb_model.predict_proba(X_test)

print("\nNaive Bayes Results")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Weighted F1:", f1_score(y_test, y_pred_nb, average='weighted'))
print("Macro F1:", f1_score(y_test, y_pred_nb, average='macro'))

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred_nb,
    target_names=['minor', 'major', 'fatal']
))

cm = confusion_matrix(y_test, y_pred_nb)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['minor', 'major', 'fatal']
)
disp.plot()
plt.title("Naive Bayes")
plt.show()

print("\nSample Probabilities")
print(pd.DataFrame(
    y_prob_nb[:10],
    columns=['P(minor)', 'P(major)', 'P(fatal)']
))

# =========================================
# MODEL 2: BAYESIAN LOGISTIC + SMOTE TOMEK
# =========================================
print("\n" + "="*60)
print("MODEL 2: BAYESIAN LOGISTIC")
print("="*60)

X_train_np = X_train.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)

y_train_np = y_train.values.astype(np.int32)
y_test_np = y_test.values.astype(np.int32)

smote_tomek = SMOTETomek(random_state=42)

X_bt, y_bt = smote_tomek.fit_resample(
    X_train_np,
    y_train_np
)

n_features = X_bt.shape[1]
n_classes = len(np.unique(y_bt))

with pm.Model() as bayes_model:

    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=1,
        shape=(n_features, n_classes)
    )

    intercept = pm.Normal(
        "intercept",
        mu=0,
        sigma=1,
        shape=(n_classes,)
    )

    logits = pm.math.dot(X_bt, beta) + intercept
    probs = pm.math.softmax(logits, axis=1)

    y_obs = pm.Categorical(
        "y_obs",
        p=probs,
        observed=y_bt
    )

    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        cores=2,
        nuts_sampler="numpyro",
        target_accept=0.95,
        random_seed=42,
        idata_kwargs={"log_likelihood": True}
    )

beta_mean = trace.posterior["beta"].mean(
    dim=("chain", "draw")
).values

intercept_mean = trace.posterior["intercept"].mean(
    dim=("chain", "draw")
).values

test_logits = np.dot(X_test_np, beta_mean) + intercept_mean

exp_logits = np.exp(
    test_logits - np.max(test_logits, axis=1, keepdims=True)
)

y_prob_bt = exp_logits / np.sum(
    exp_logits,
    axis=1,
    keepdims=True
)

y_pred_bt = np.argmax(y_prob_bt, axis=1)

print("\nBayesian Logistic Results")
print("Accuracy:", accuracy_score(y_test_np, y_pred_bt))
print("Weighted F1:", f1_score(y_test_np, y_pred_bt, average='weighted'))
print("Macro F1:", f1_score(y_test_np, y_pred_bt, average='macro'))

print("\nClassification Report")
print(classification_report(
    y_test_np,
    y_pred_bt,
    target_names=['minor', 'major', 'fatal']
))

waic = az.waic(trace)
print("\nWAIC:", waic.elpd_waic)

cm = confusion_matrix(y_test_np, y_pred_bt)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['minor', 'major', 'fatal']
)
disp.plot()
plt.title("Bayesian Logistic")
plt.show()

# =========================================
# MODEL 3: PRIOR SENSITIVITY ANALYSIS
# =========================================
print("\n" + "="*60)
print("MODEL 3: PRIOR SENSITIVITY")
print("="*60)

prior_options = {
    "weak_prior": 1,
    "medium_prior": 2,
    "non_informative": 5
}

results = {}

for prior_name, sigma_prior in prior_options.items():

    print(f"\nRunning {prior_name}")

    with pm.Model() as model:

        beta = pm.Normal(
            "beta",
            mu=0,
            sigma=sigma_prior,
            shape=(n_features, n_classes)
        )

        intercept = pm.Normal(
            "intercept",
            mu=0,
            sigma=sigma_prior,
            shape=(n_classes,)
        )

        logits = pm.math.dot(X_bt, beta) + intercept
        probs = pm.math.softmax(logits, axis=1)

        y_obs = pm.Categorical(
            "y_obs",
            p=probs,
            observed=y_bt
        )

        trace = pm.sample(
            draws=1000,
            tune=1000,
            chains=2,
            cores=2,
            nuts_sampler="numpyro",
            target_accept=0.95,
            random_seed=42,
            idata_kwargs={"log_likelihood": True}
        )

    beta_mean = trace.posterior["beta"].mean(
        dim=("chain", "draw")
    ).values

    intercept_mean = trace.posterior["intercept"].mean(
        dim=("chain", "draw")
    ).values

    logits_test = np.dot(X_test_np, beta_mean) + intercept_mean

    exp_logits = np.exp(
        logits_test - np.max(logits_test, axis=1, keepdims=True)
    )

    probs_test = exp_logits / np.sum(
        exp_logits,
        axis=1,
        keepdims=True
    )

    preds = np.argmax(probs_test, axis=1)

    acc = accuracy_score(y_test_np, preds)
    wf1 = f1_score(y_test_np, preds, average='weighted')
    mf1 = f1_score(y_test_np, preds, average='macro')

    waic = az.waic(trace)

    results[prior_name] = {
        "Accuracy": acc,
        "Weighted F1": wf1,
        "Macro F1": mf1,
        "WAIC": waic.elpd_waic
    }

    print("\nClassification Report")
    print(classification_report(
        y_test_np,
        preds,
        target_names=['minor', 'major', 'fatal']
    ))

    cm = confusion_matrix(y_test_np, preds)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['minor', 'major', 'fatal']
    )
    disp.plot()
    plt.title(f"Confusion Matrix - {prior_name}")
    plt.show()

# =========================================
# FINAL COMPARISON
# =========================================
comparison_df = pd.DataFrame(results).T

print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)

print(comparison_df.sort_values(
    "Weighted F1",
    ascending=False
))

In [ ]:
# =========================================
# Bayesian Clustering
# =========================================


# =========================================
# STEP 1: IMPORTS
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import jax

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import BayesianGaussianMixture
from sklearn.metrics import silhouette_score

print("JAX Backend:", jax.default_backend())
print("Devices:", jax.devices())

# =========================================
# STEP 2: LOAD DATA
# =========================================
path = "../data/indian_roads_dataset.csv"
df = pd.read_csv(path)

# =========================================
# STEP 3: PREPROCESSING
# =========================================
selected_columns = [
    'time', 'date',
    'hour', 'day_of_week', 'is_weekend',
    'road_type', 'lanes',
    'traffic_signal', 'weather', 'visibility',
    'traffic_density', 'is_peak_hour', 'festival'
]

df_processed = df[selected_columns].copy()

df_processed['festival'] = df_processed['festival'].notna().astype(int)
df_processed['date'] = pd.to_datetime(df_processed['date'])
df_processed['month'] = df_processed['date'].dt.month

day_map = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2,
    'Thursday': 3, 'Friday': 4,
    'Saturday': 5, 'Sunday': 6
}

df_processed['weekday_num'] = df_processed['day_of_week'].map(day_map)

one_hot_cols = [
    'road_type',
    'traffic_signal',
    'weather',
    'visibility',
    'traffic_density'
]

df_processed = pd.get_dummies(
    df_processed,
    columns=one_hot_cols,
    drop_first=True
)

scale_cols = ['lanes', 'month', 'weekday_num']

scaler = StandardScaler()
df_processed[scale_cols] = scaler.fit_transform(df_processed[scale_cols])

df_processed.drop(
    columns=['time', 'date', 'day_of_week'],
    inplace=True
)

print("Processed shape:", df_processed.shape)

# =========================================
# STEP 4: PCA
# =========================================
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(df_processed)

print("Original shape :", df_processed.shape)
print("Reduced shape  :", X_pca.shape)
print("Variance kept  :", round(
    pca.explained_variance_ratio_.sum() * 100, 2
), "%")

X = X_pca.astype(np.float32)

# =========================================
# PCA FEATURE IMPORTANCE GRAPH
# =========================================
loadings = pd.DataFrame(
    pca.components_.T,
    index=df_processed.columns
)

feature_importance = pd.DataFrame({
    "Feature": df_processed.columns,
    "Importance": np.sum(np.abs(loadings.values), axis=1)
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

top_features = feature_importance.head(10)

plt.figure(figsize=(8,5))
plt.bar(top_features["Feature"], top_features["Importance"])
plt.xticks(rotation=45, ha='right')
plt.ylabel("Importance Score")
plt.title("Top 10 Important Features from PCA")
plt.tight_layout()
plt.show()

# =========================================
# MODEL 1: BGMM
# =========================================
bgmm = BayesianGaussianMixture(
    n_components=3,
    covariance_type='full',
    weight_concentration_prior_type='dirichlet_process',
    weight_concentration_prior=0.01,
    max_iter=500,
    n_init=5,
    random_state=42
)

bgmm.fit(X)
bgmm_labels = bgmm.predict(X)

sil_bgmm = silhouette_score(X, bgmm_labels)

print("BGMM Silhouette:", round(sil_bgmm, 4))

# BGMM SCATTER GRAPH
plt.figure(figsize=(7,5))
plt.scatter(X[:, 0], X[:, 1], c=bgmm_labels)
plt.title(f"BGMM Clusters | Sil={sil_bgmm:.3f}")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.show()

# BGMM CLUSTER SIZE GRAPH
plt.figure(figsize=(6,4))
pd.Series(bgmm_labels).value_counts().sort_index().plot(kind='bar')
plt.title("BGMM Cluster Counts")
plt.xlabel("Cluster")
plt.ylabel("Count")
plt.grid(True)
plt.show()

# =========================================
# MODEL 2: Generative BAYESIAN MIXTURE
# =========================================
K = 3
n_features = X.shape[1]

with pm.Model() as model:

    weights = pm.Dirichlet("weights", a=np.ones(K))

    means = pm.Normal(
        "means",
        mu=0,
        sigma=5,
        shape=(K, n_features)
    )

    sigma = pm.HalfNormal(
        "sigma",
        sigma=2,
        shape=(K, n_features)
    )

    components = [
        pm.Normal.dist(
            mu=means[k],
            sigma=sigma[k],
            shape=n_features
        )
        for k in range(K)
    ]

    obs = pm.Mixture(
        "obs",
        w=weights,
        comp_dists=components,
        observed=X
    )

    trace = pm.sampling.jax.sample_numpyro_nuts(
        draws=1000,
        tune=1000,
        chains=2,
        target_accept=0.9,
        chain_method="vectorized",
        random_seed=42
    )

# TRACE PLOTS
az.plot_trace(trace)
plt.show()

# POSTERIOR WEIGHTS GRAPH
posterior_weights = trace.posterior["weights"].mean(
    dim=["chain", "draw"]
).values

plt.figure(figsize=(6,4))
plt.bar(range(K), posterior_weights)
plt.title("Posterior Cluster Weights")
plt.xlabel("Cluster")
plt.ylabel("Weight")
plt.grid(True)
plt.show()

# CLUSTER ASSIGNMENT
posterior_means = trace.posterior["means"].mean(
    dim=["chain", "draw"]
).values

distances = np.linalg.norm(
    X[:, None, :] - posterior_means[None, :, :],
    axis=2
)

pymc_labels = np.argmin(distances, axis=1)

sil_pymc = silhouette_score(X, pymc_labels)

print("PyMC Silhouette:", round(sil_pymc, 4))

# PYMC SCATTER GRAPH
plt.figure(figsize=(7,5))
plt.scatter(X[:, 0], X[:, 1], c=pymc_labels)
plt.title(f"PyMC Bayesian Clusters | Sil={sil_pymc:.3f}")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.show()

# PYMC CLUSTER COUNTS GRAPH
plt.figure(figsize=(6,4))
pd.Series(pymc_labels).value_counts().sort_index().plot(kind='bar')
plt.title("PyMC Cluster Counts")
plt.xlabel("Cluster")
plt.ylabel("Count")
plt.grid(True)
plt.show()

# =========================================
# MODEL COMPARISON GRAPH
# =========================================
plt.figure(figsize=(6,4))
plt.bar(
    ["BGMM", "PyMC"],
    [sil_bgmm, sil_pymc]
)
plt.title("Model Comparison by Silhouette Score")
plt.ylabel("Score")
plt.grid(True)
plt.show()